# Kwame's Code

## Merge Portions of NBA Data Together


In [3]:
# ▸ Cell 1 ─────────────────────────────────────────────────────────────
# merge_data.ipynb  – v3
# Build a two-row-per-game DataFrame from Kaggle's game.csv + line_score.csv
#   • renames *_home / *_away → *_main / *_opp  in BOTH files (pre-merge)
#   • merges on game_id
#   • duplicates each game with roles flipped (iscopy = 0 / 1)
# ---------------------------------------------------------------------
import pandas as pd
from pathlib import Path

# Notebook lives in ./notebooks/
DATA_DIR       = Path("../data")
RAW_DIR        = DATA_DIR / "raw"
CSV_DIR        = RAW_DIR  / "csv"
PROCESSED_DIR  = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

print("✓ Directory layout checked")

✓ Directory layout checked


In [4]:
# ▸ Cell 2 ─────────────────────────────────────────────────────────────
# 1. Load *all* columns
# ---------------------------------------------------------------------
game_df = pd.read_csv(CSV_DIR / "game.csv")          # every column
line_df = pd.read_csv(CSV_DIR / "line_score.csv")    # every column

print(f"game_df shape : {game_df.shape}")
print(f"line_df shape : {line_df.shape}")

game_df shape : (65698, 55)
line_df shape : (58053, 43)


In [5]:
# ▸ Cell 3 ─────────────────────────────────────────────────────────────
# 2. Rename *_home / *_away columns → *_main / *_opp    (in BOTH DataFrames)
# ---------------------------------------------------------------------
def rename_home_away(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with *_home / *_away renamed to *_main / *_opp."""
    home_cols = [c for c in df.columns if c.endswith("_home")]
    away_cols = [c for c in df.columns if c.endswith("_away")]
    rename_map = {c: f"{c[:-5]}_main" for c in home_cols} | \
                 {c: f"{c[:-5]}_opp"  for c in away_cols}
    return df.rename(columns=rename_map)

line_df = rename_home_away(line_df)
game_df = rename_home_away(game_df)

print("✓ Renamed suffixes in both tables")

✓ Renamed suffixes in both tables


In [6]:
# ▸ Cell 4 ─────────────────────────────────────────────────────────────
# 3. Merge on game_id  (line_df = “left”, game_df fields get *_game suffix)
# ---------------------------------------------------------------------
wide = (
    line_df
      .merge(game_df, on="game_id", how="left", suffixes=("", "_game"))
)

missing = wide["game_date"].isna().sum()
print(f"merge → {wide.shape} rows   |   unmatched game_ids = {missing}")

merge → (58149, 97) rows   |   unmatched game_ids = 0


In [7]:
# ▸ Cell 5 ─────────────────────────────────────────────────────────────
# 4. Build two orientations
#    • iscopy = 0 : home team is MAIN  (already true)
#    • iscopy = 1 : away team is MAIN  (swap values in *_main / *_opp pairs)
# ---------------------------------------------------------------------
home_main = wide.assign(iscopy=0)          # orientation 1

away_main = wide.copy()                    # orientation 2 (to be flipped)
# Identify every column whose *name* marks it as "main"
main_cols = [c for c in wide.columns if "_main" in c]

# Swap each *_main ↔ *_opp partner (handles *_main_game too)
for main_col in main_cols:
    opp_col = main_col.replace("_main", "_opp")
    if opp_col in away_main.columns:
        away_main[[main_col, opp_col]] = away_main[[opp_col, main_col]].to_numpy()

away_main = away_main.assign(iscopy=1)

# Stack the two tables
full_nba = pd.concat([home_main, away_main], ignore_index=True)

print(f"✓ full_nba shape : {full_nba.shape}  (expected 2 × line_df rows)")

✓ full_nba shape : (116298, 98)  (expected 2 × line_df rows)


In [8]:
# ▸ Cell 6 ─────────────────────────────────────────────────────────────
# 5. Quick peek
# ---------------------------------------------------------------------
show_cols = [c for c in full_nba.columns if c.endswith(("_main", "_opp"))][:10]
print("Sample paired columns:", show_cols)
full_nba.head()

Sample paired columns: ['team_id_main', 'team_abbreviation_main', 'team_city_name_main', 'team_nickname_main', 'team_wins_losses_main', 'pts_qtr1_main', 'pts_qtr2_main', 'pts_qtr3_main', 'pts_qtr4_main', 'pts_ot1_main']


,game_date_est,game_sequence,game_id,team_id_main,team_abbreviation_main,team_city_name_main,team_nickname_main,team_wins_losses_main,pts_qtr1_main,pts_qtr2_main,...,ast_opp,stl_opp,blk_opp,tov_opp,pf_opp,pts_opp_game,plus_minus_opp,video_available_opp,season_type,iscopy
0,1946-11-01 00:00:00,NaN,24600001,1610610035,HUS,Toronto,Huskies,-,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,68.0,2,0,Regular Season,0
1,1946-11-02 00:00:00,NaN,24600003,1610610034,BOM,St. Louis,Bombers,-,16.0,16.0,...,NaN,NaN,NaN,NaN,25.0,51.0,-5,0,Regular Season,0
2,1946-11-02 00:00:00,NaN,24600002,1610612738,BOS,Boston,Celtics,-,10.0,16.0,...,NaN,NaN,NaN,NaN,NaN,53.0,-6,0,Regular Season,0
3,1946-11-02 00:00:00,NaN,24600004,1610610025,CHS,Chicago,Stags,-,NaN,NaN,...,NaN,NaN,NaN,NaN,22.0,47.0,-16,0,Regular Season,0
4,1946-11-02 00:00:00,NaN,24600005,1610610036,WAS,Washington,Capitols,-,21.0,4.0,...,NaN,NaN,NaN,NaN,NaN,50.0,17,0,Regular Season,0


In [9]:
# ▸ Cell 7 ─────────────────────────────────────────────────────────────
# Remove any *_game columns (they’re duplicates from game.csv)
full_nba = full_nba.loc[:, ~full_nba.columns.str.endswith("_game")]
# -----------------------------------------------------------------------

In [10]:
# ▸ Cell 8 ──────────────────────────────────────────────────────────
# Normalise the date field: keep only calendar date, drop old columns
# --------------------------------------------------------------------
# 1. Build the new date_game column from game_date_est
full_nba["date_game"] = pd.to_datetime(full_nba["game_date_est"]).dt.date

# 2. Drop the superseded columns
full_nba = full_nba.drop(columns=["game_date_est", "game_date"], errors="ignore")

In [11]:
# ▸ Cell 9 ──────────────────────────────────────────────────────────
# Re-order columns:
#   1) game_id
#   2) date_game
#   3) *_main  columns (in the order they already appear)
#   4) *_opp   columns
#   5) everything else (e.g. iscopy)
# --------------------------------------------------------------------
main_cols  = [c for c in full_nba.columns if c.endswith("_main")]
opp_cols   = [c for c in full_nba.columns if c.endswith("_opp")]

# Preserve original order for any remaining columns
other_cols = [
    c for c in full_nba.columns
    if c not in {"game_id", "date_game"} | set(main_cols) | set(opp_cols)
]

new_order = ["game_id", "date_game"] + main_cols + opp_cols + other_cols
full_nba = full_nba[new_order]

print("✓ Columns reordered")

✓ Columns reordered


In [12]:
# ▸ Cell 10 ──────────────────────────────────────────────────────────
full_nba.head()

,game_id,date_game,team_id_main,team_abbreviation_main,team_city_name_main,team_nickname_main,team_wins_losses_main,pts_qtr1_main,pts_qtr2_main,pts_qtr3_main,...,blk_opp,tov_opp,pf_opp,plus_minus_opp,video_available_opp,game_sequence,season_id,min,season_type,iscopy
0,24600001,1946-11-01,1610610035,HUS,Toronto,Huskies,-,NaN,NaN,NaN,...,NaN,NaN,NaN,2,0,NaN,21946,0,Regular Season,0
1,24600003,1946-11-02,1610610034,BOM,St. Louis,Bombers,-,16.0,16.0,18.0,...,NaN,NaN,25.0,-5,0,NaN,21946,0,Regular Season,0
2,24600002,1946-11-02,1610612738,BOS,Boston,Celtics,-,10.0,16.0,14.0,...,NaN,NaN,NaN,-6,0,NaN,21946,0,Regular Season,0
3,24600004,1946-11-02,1610610025,CHS,Chicago,Stags,-,NaN,NaN,NaN,...,NaN,NaN,22.0,-16,0,NaN,21946,0,Regular Season,0
4,24600005,1946-11-02,1610610036,WAS,Washington,Capitols,-,21.0,4.0,12.0,...,NaN,NaN,NaN,17,0,NaN,21946,0,Regular Season,0


In [13]:
# ▸ Cell 11 ──────────────────────────────────────────────────────────
full_nba.columns

Index(['game_id', 'date_game', 'team_id_main', 'team_abbreviation_main',
       'team_city_name_main', 'team_nickname_main', 'team_wins_losses_main',
       'pts_qtr1_main', 'pts_qtr2_main', 'pts_qtr3_main', 'pts_qtr4_main',
       'pts_ot1_main', 'pts_ot2_main', 'pts_ot3_main', 'pts_ot4_main',
       'pts_ot5_main', 'pts_ot6_main', 'pts_ot7_main', 'pts_ot8_main',
       'pts_ot9_main', 'pts_ot10_main', 'pts_main', 'team_name_main',
       'matchup_main', 'wl_main', 'fgm_main', 'fga_main', 'fg_pct_main',
       'fg3m_main', 'fg3a_main', 'fg3_pct_main', 'ftm_main', 'fta_main',
       'ft_pct_main', 'oreb_main', 'dreb_main', 'reb_main', 'ast_main',
       'stl_main', 'blk_main', 'tov_main', 'pf_main', 'plus_minus_main',
       'video_available_main', 'team_id_opp', 'team_abbreviation_opp',
       'team_city_name_opp', 'team_nickname_opp', 'team_wins_losses_opp',
       'pts_qtr1_opp', 'pts_qtr2_opp', 'pts_qtr3_opp', 'pts_qtr4_opp',
       'pts_ot1_opp', 'pts_ot2_opp', 'pts_ot3_opp', 'pts

In [14]:
# ▸ Cell 12 ─────────────────────────────────────────────────────────────
# Save the tidy full_nba table   (choose the format you prefer)
# ---------------------------------------------------------------------
out_path = PROCESSED_DIR / "full_nba.csv"
full_nba.to_csv(out_path, index=False)

print(f"✓ Saved → {out_path}") 

✓ Saved → ../data/processed/full_nba.csv


# Brendan's Code

In [1]:
import pandas as pd
from pathlib import Path
import io, requests

All columns from the kaggle dataset:

`Index(['season_id', 'team_id_home', 'team_abbreviation_home', 'team_name_home',
       'game_id', 'game_date', 'matchup_home', 'wl_home', 'min', 'fgm_home',
       'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home',
       'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home',
       'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home',
       'pts_home', 'plus_minus_home', 'video_available_home', 'team_id_away',
       'team_abbreviation_away', 'team_name_away', 'matchup_away', 'wl_away',
       'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away',
       'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away',
       'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away',
       'pf_away', 'pts_away', 'plus_minus_away', 'video_available_away',
       'season_type'],
      dtype='object')`

In [114]:
# ---------------------------------------------------------------------
# 1.  Load Kaggle game metadata (date, season, id)
# ---------------------------------------------------------------------
CSV_DIR   = Path("data/csv")
DATA_DIR  = Path("data")

game_df = pd.read_csv(CSV_DIR / "game.csv",
                      usecols=["game_id", "game_date", "season_id", "wl_home", 'fgm_home',
       'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home',
       'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home',
       'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home',
       'plus_minus_home', 'wl_away', 
       'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away',
       'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away',
       'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away',
       'pf_away', 'pts_away', 'plus_minus_away'])
game_df["date_game"] = pd.to_datetime(game_df["game_date"]).dt.date

In [115]:
# ---------------------------------------------------------------------
# 2.  Load Kaggle line scores (already 1 row per TEAM per GAME)
# ---------------------------------------------------------------------
line_df = pd.read_csv(CSV_DIR / "line_score.csv",
                      usecols=["game_id", "team_abbreviation_home", "team_abbreviation_away", "pts_home"])

# bring in the date + season
kaggle_long = line_df.merge(game_df,
                            on="game_id", how="inner")

In [116]:
# ---------------------------------------------------------------------
# 3.  Load & reshape FiveThirtyEight Elo → long format
# ---------------------------------------------------------------------
elo_538_raw = pd.read_csv(DATA_DIR / "nbaallelo.csv")
elo_538_raw["date_game"] = pd.to_datetime(elo_538_raw["date_game"]).dt.date

elo_team1 = (
    elo_538_raw[["date_game", "team_id", "elo_i", "elo_n", "is_playoffs"]]
      .rename(columns={"team_id": "TEAM_ABBREVIATION",
                       "elo_i": "elo_pre_538",
                       "elo_n": "elo_post_538"})
)
elo_team2 = (
    elo_538_raw[["date_game", "opp_id", "opp_elo_i", "opp_elo_n", "is_playoffs"]]
      .rename(columns={"opp_id": "TEAM_ABBREVIATION",
                       "opp_elo_i": "elo_pre_538",
                       "opp_elo_n": "elo_post_538"})
)

elo_538_long = pd.concat([elo_team1, elo_team2], ignore_index=True)

In [117]:
# ---------------------------------------------------------------------
# 4.  Reconcile historical team codes (minimal starter dict)
# ---------------------------------------------------------------------
alias = {
    "NJN": "BKN", "BRK": "BKN",   # Nets
    "NOH": "NOP", "NOK": "NOP",   # Pelicans
    "CHH": "CHA",                 # Old Hornets
    "SEA": "OKC",                 # Sonics → Thunder
}
kaggle_long["team_abbreviation_home"] = kaggle_long["team_abbreviation_home"].replace(alias)

kaggle_long.rename(columns={"team_abbreviation_home": "TEAM_ABBREVIATION"}, inplace=True)

elo_538_long.rename(columns={"team_id": "TEAM_ABBREVIATION"}, inplace=True)
elo_538_long["TEAM_ABBREVIATION"] = elo_538_long["TEAM_ABBREVIATION"].replace(alias)


In [118]:
# ---------------------------------------------------------------------
# 5.  Merge on date + franchise code
# ---------------------------------------------------------------------

# TO DO: Merge in a way that keeps both teams that played the game
# TO do: The source of the problem is that Kaggle has one row per game,
# and FiveThirtyEight has one row per team per game. Need to find whether Kaggle has both teams listed as separate rows before they were put into long format.
# Problem is that we merged based on date and team code 
merged = kaggle_long.merge(
    elo_538_long,
    on=["date_game", "TEAM_ABBREVIATION"],
    how="inner"
)

In [119]:
# Currently, the merged DataFrame has duplicate rows for each game. We need to take each duplicate and swap the columns for the two teams, so that we have a single row per game with both teams' data.

# Taking every other row and swapping the columns for the two teams
copy_1_df = merged[::2]

# Swap 'A' and 'B' column names
swap_mapping = {'TEAM_ABBREVIATION': 'team_abbreviation_away_temp', 'pts_home':'pts_away_temp', 'wl_home':'wl_away_temp',
                'fgm_home':'fgm_away_temp', 'fga_home':'fga_away_temp', 'fg_pct_home':'fg_pct_away_temp', 
                'fg3m_home':'fg3m_away_temp', 'fg3a_home':'fg3a_away_temp', 'fg3_pct_home':'fg3_pct_away_temp',
       'ftm_home':'ftm_away_temp', 'fta_home':'fta_away_temp', 'ft_pct_home':'ft_pct_away_temp', 'oreb_home':'oreb_away_temp', 'dreb_home': 'dreb_away_temp',
       'reb_home':'reb_away_temp', 'ast_home':'ast_away_temp', 'stl_home':'stl_away_temp', 'blk_home':'blk_away_temp',
       'tov_home':'tov_away_temp', 'pf_home':'pf_away_temp', 'plus_minus_home':'plus_minus_away_temp',
       # away names also being swapped
       'team_abbreviation_away':'TEAM_ABBREVIATION_temp', 'pts_away':'pts_home_temp', 
       'wl_away':'wl_home_temp', 'fgm_away':'fgm_home_temp', 'fga_away':'fga_home_temp', 'fg_pct_away':'fg_pct_home_temp',
       'fg3m_away':'fg3m_home_temp', 'fg3a_away':'fg3a_home_temp', 
       'fg3_pct_away':'fg3_pct_home_temp', 'ftm_away':'ftm_home_temp', 'fta_away':'fta_home_temp', 'ft_pct_away':'ft_pct_home_temp', 'oreb_away':'oreb_home_temp',
       'dreb_away':'dreb_home_temp', 'reb_away':'reb_home_temp', 'ast_away':'ast_home_temp', 'stl_away':'stl_home_temp', 
       'blk_away':'blk_home_temp', 'tov_away':'tov_home_temp', 'pf_away':'pf_home_temp', 'plus_minus_away':"plus_minus_home_temp"} # Using temporary names
copy_1_df.rename(columns=swap_mapping, inplace=True)

# Now rename the temporary names to their final desired names
final_rename_mapping = {'team_abbreviation_away_temp':'team_abbreviation_away', 'pts_away_temp': 'pts_away', 
                        'wl_away_temp': 'wl_away', 'fgm_away_temp': 'fgm_away', 'fga_away_temp': 'fga_away',
                        'fg_pct_away_temp': 'fg_pct_away', 'fg3m_away_temp': 'fg3m_away', 'fg3a_away_temp': 'fg3a_away',
                        'fg3_pct_away_temp': 'fg3_pct_away', 'ftm_away_temp': 'ftm_away', 'fta_away_temp': 'fta_away',
                        'ft_pct_away_temp': 'ft_pct_away', 'oreb_away_temp': 'oreb_away', 'dreb_away_temp': 'dreb_away',
                        'reb_away_temp': 'reb_away', 'ast_away_temp': 'ast_away', 'stl_away_temp': 'stl_away', 
                        'blk_away_temp': 'blk_away', 'tov_away_temp': 'tov_away', 'pf_away_temp': 'pf_away', 'plus_minus_away_temp':'plus_minus_away',
                        # Now swap the home team columns back to their original names, with the away and home teams swapped
                        "TEAM_ABBREVIATION_temp": "TEAM_ABBREVIATION", 'wl_home_temp': "wl_home",
                        "pts_home_temp": "pts_home", "fgm_home_temp": "fgm_home", "fga_home_temp": "fga_home",
                        "fg_pct_home_temp": "fg_pct_home", "fg3m_home_temp": "fg3m_home", "fg3a_home_temp": "fg3a_home",
                        "fg3_pct_home_temp": "fg3_pct_home", "ftm_home_temp": "ftm_home", "fta_home_temp": "fta_home",
                        "ft_pct_home_temp": "ft_pct_home", "oreb_home_temp": "oreb_home", "dreb_home_temp": "dreb_home",
                        "reb_home_temp": "reb_home", "ast_home_temp": "ast_home", "stl_home_temp": "stl_home",
                        "blk_home_temp": "blk_home", "tov_home_temp": "tov_home", "pf_home_temp": "pf_home", "plus_minus_home_temp": "plus_minus_home"
                        }
copy_1_df.rename(columns=final_rename_mapping, inplace=True)


/var/folders/xq/z90fxfv513nfkp28682x0phh0000gn/T/ipykernel_64177/355580081.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  copy_1_df.rename(columns=swap_mapping, inplace=True)
/var/folders/xq/z90fxfv513nfkp28682x0phh0000gn/T/ipykernel_64177/355580081.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  copy_1_df.rename(columns=final_rename_mapping, inplace=True)


In [120]:
copy_2_df = merged[1::2]
final_merged = pd.concat([copy_1_df, copy_2_df], axis=0, ignore_index=True)


In [124]:
final_merged.head(-10)
final_merged[final_merged['game_id'] == 24600002]

,game_id,team_abbreviation_away,pts_away,TEAM_ABBREVIATION,season_id,game_date,wl_away,fgm_away,fga_away,fg_pct_away,...,stl_home,blk_home,tov_home,pf_home,pts_home,plus_minus_home,date_game,elo_pre_538,elo_post_538,is_playoffs
0,24600002,BOS,53.0,PRO,21946,1946-11-02 00:00:00,W,21.0,NaN,NaN,...,NaN,NaN,NaN,NaN,53.0,-6,1946-11-02,1300.0,1294.8458,0
40366,24600002,PRO,53.0,BOS,21946,1946-11-02 00:00:00,L,21.0,NaN,NaN,...,NaN,NaN,NaN,NaN,53.0,6,1946-11-02,1300.0,1294.8458,0


In [122]:
# ---------------------------------------------------------------------
# 6.  Save the merged DataFrame to a CSV file
# ---------------------------------------------------------------------
final_merged.to_csv("data/raw/combined_kaggle_538_elo.csv", index=False)

In [ ]:
# TO DO: Make sure ELO is different for diffrent teams in the same game